# Automated Alignment

This describes how to run automated aligners with alignment data in Scripture Burrito format. This notebook addresses 
* generating aligner input in "fast_align" (piped) format
* setting up `eflomal`
* running `eflomal` to produce output.

See [Scoring Alignment Data](31ScoringAlignmentData.ipynb) for scoring the results. 

## Contents

* [Naming Conventions](#Naming-Conventions)
    * [Aligner Input](#Aligner-Input)
    * [Aligner Output](#Aligner-Output)
* [Generating Aligner Input](#Generating-Aligner-Input)
    * [Generating Lemmatized Data](#Generating-Lemmatized-Data)
    * [Generating Partial Data](#Generating-Partial-Data)
* [Running Automated Alignment](#Running-Automated-Alignment)
    * [Option: Different Input](#Option:-Different-Input)
    * [Option: Using Priors](#Option:-Using-Priors)
    * [Option: Aligning Partial Data](#Option:-Aligning-Partial-Data)
* [Regenerating Burrito Data](#Regenerating-Burrito-Data)
    * [Regenerating Partial Burrito Data](#Regenerating-Partial-Burrito-Data)
    * [Regenerating Burrito Data from Partial Data](#Regenerating-Burrito-Data-from-Partial-Data)
* [Next Steps](#Next-Steps)


## Process Overview

Here's a visual overview of the whole process. 
```mermaid
flowchart TB
    Source[Source TSV 🌯]
    Target[Target TSV 🌯]
    Gold[Reference Alignment 🌯]
    AlGroup[in-memory AlignmentGroup instance]
    Exp[Hypothesized Alignment 🌯]
    MGR[autoalign/pharoah.py/PharoahMapper]
    Piped[Piped-format input to aligner]
    Pharaoh[Pharaoh-format output from aligner]
    Scorer[autoalign/scorer.py/Scorer]
    Verse[Matrix for a single verse]
    Group[Score for a regexp-defined group of verses]
    All[Score for all verses]

    Source --> MGR
    Target --> MGR
    Gold -.-> MGR
    MGR --write_piped()--> Piped --(external aligner)--> Pharaoh
    Pharaoh --read_pharaoh()--> AlGroup
    AlGroup --burrito/alignments/write_alignment_group()--> Exp
    Source --> Scorer
    Target --> Scorer
    Gold --> Scorer
    Exp --> Scorer
    Scorer --verse_dataframe()--> Verse
    Scorer --score_group()--> Group
    Scorer --score_all()--> All


```

<!-- (You can view this in VS Code with the [Markdown Preview Mermaid Support](https://marketplace.visualstudio.com/items?itemName=bierner.markdown-mermaid) extension, then select Open Preview on the top right or Cmd-k v.)
-->

## Naming Conventions

There are numerous files associated with running automated aligners, and we anticipate running numerous experimental conditions with different input, different aligners, different scoring, etc. To manage these details, we adopt the following conventions. 

Parameters:
* `$GITROOT` below refers to the root of your Github files for https://github.com/Clear-Bible/, which are assumed to all live under the same root. 
* `$LANG` is the ISO-639 code for the language in question.
* `$SOURCEID` is abbreviation for the source text, typically `SBLGNT` or `WLCM`.
* `$TARGETID` is the abbreviation translation being aligned.

### Aligner Input

This data should change infrequently, and isn't typically part of most experimental conditions. It is therefore staged in `$GITROOT/autoalignment/data/$LANG/$TARGETID`. The basic text version is named `$SOURCEID-$TARGETID.piped.txt` , with modifiers added to the end of the stem: for example, `$SOURCEID-$TARGETID-lemma.piped.txt` for files using lemma rather than surface text for source. 

If you're aligning a partial text (e.g. just one or more Bible books), use some modifier that clearly expresses the scope of the data. 

Example: lemmatized source text for SBLGNT and BSB is found in `$GITROOT/autoalignment/data/eng/BSB/SBLGNT-BSB-lemma.piped.txt`. 

### Aligner Output

Aligner output should go in the language-specific repository, in the exp/`$TARGETID` directory. 
The hypothesis data that is output from the aligner represents a specific experimental condition, and is therefore best captured in a single directory that is named to indicate the condition, referred to here as the "condition directory" (`$CONDITIONDIR`). There may be too many parameters associated with the condition to bundle into the condition directory name, so choose the most distinctive name. In a pinch, append date stamps to make names unique (“foo-20241203" vs “foo-20241204”). 

Example: the condition directory for a BSB experiment on SBLGNT with `eflomal` on lemmatized data might be named `SBLGNT-eflomal-lemmatized`. 

The files produced in this directory depend on the alignment algorithm. For `eflomal`, that might include files like:
* `forward.pharaoh.txt` and `reverse.pharaoh.txt`: the forward and reverse passes for alignment (pharaoh format)
* `pharaoh.txt`: the symmetrized pharaoh-format output
* `log_<DATE&TIME>.txt`: the log file from running the processes
* `$SOURCEID-$TARGETID-eflomal.json`: the output converted to Burrito format (the *hypothesis* data)

These directories and files should be commited to Git for reproducability. It's good practice to remove directories that have been superseded and are no longer relevant, to reduce clutter. 

## Generating Aligner Input

`eflomal` takes "piped" (`fast_align`) format data as input: one line per verse, source text string + "|||" + target text string. Both source and target text must be provided: if one is not available, use some placeholder text like "MISSING". 

In [1]:
from biblealignlib.burrito import CLEARROOT, AlignmentSet
from biblealignlib.autoalign import mapper, reader, writer, eflomal
# your local copy of alignments-eng/data
(targetlang, targetid, sourceid) = ("eng", "BSB", "SBLGNT")
# bundled parameters for downstream processes
alset = AlignmentSet(targetlanguage=targetlang,
        targetid=targetid,
        sourceid=sourceid,
        langdatapath=(CLEARROOT / f"alignments-{targetlang}/data"))


In [2]:
pm = mapper.PharaohMapper(alset)
mapping = pm.bcv["mappings"]["41004003"]
sourcetokenattr: str = "text"
[getattr(item, sourcetokenattr) for item, _ in mapping.source_pairs]


        - sourcepath: /Users/sboisen/git/Clear-Bible/Alignments/data/sources/SBLGNT.tsv
        - targetpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/targets/BSB/nt_BSB.tsv
        - alignmentpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/alignments/BSB/SBLGNT-BSB-manual.json
        - tomlpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/alignments/BSB/SBLGNT-BSB-manual.toml
        


['Ἀκούετε', 'ἰδοὺ', 'ἐξῆλθεν', 'ὁ', 'σπείρων', 'σπεῖραι']

In [3]:
mapping

<CorpusMapping: 41004003>

In [4]:
pw = writer.PharaohWriter(pm)
# directory for output
pw.pipedpath

PosixPath('/Users/sboisen/git/Clear-Bible/autoalignment/data/eng/BSB/SBLGNT-BSB.piped.txt')

**Caution**: the aligner input data is shared. Generating data in the `autoalignment` repo and committing it may therefore affect everyone else's experiments. Depending on the changes, others might need to know so they can re-establish any baselines. 

In [5]:
# write pharaoh data for automated alignment
# by default, file is named f"{alset.identifier}.piped.txt"
# ---------------------------------------------------------
# only run this if you want to update the actual data
# pw.write_piped()

### Generating Lemmatized Data

These are the conventions for creating lemmatized data. 

In [6]:
# use a different name for the piped file
pw = writer.PharaohWriter(pm, pipedname="SBLGNT-BSB-lemma.piped.txt")
# directory for output
pw.pipedpath

PosixPath('/Users/sboisen/git/Clear-Bible/autoalignment/data/eng/BSB/SBLGNT-BSB-lemma.piped.txt')

In [7]:
# use source lemma attributes
# ---------------------------------------------------------
# only run this if you want to update the actual data
# pw.write_piped(sourcetokenattr="lemma")

### Generating Partial Data

You may want to generate only part of a corpus if you only have partial gold standard data (perhaps because manual alignment is still in progress). To do this, first write partial data.

In [8]:
# just Mark
ppw = writer.PharaohPartialWriter(pm, "41001001", "41016020")
# then pw.write_piped

## Running Automated Alignment

First, ensure you have an `eflomal` executable, following the directions [here](https://github.com/Clear-Bible/internal-Alignments?tab=readme-ov-file#eflomal). You only need to do this once. 

This makes the shell executables `eflomal-align` and `eflomal-priors` available to your poetry environment for `internal-alignments`. The `Eflomal` class in `src/autoalign/eflomal.py` bundles up naming conventions, and is therefore an easy way to run the processes with the right parameters. You need to provide the `$CONDITIONDIR` for output. 

In [9]:
from biblealignlib.autoalign import eflomal

In [10]:
# this will create the directory if it doesn't already exist
# BUG: running eflomal doesn't yet work in biblealignlib
condition = "notebookcheck"
efinst = eflomal.Eflomal(alset, condition)
efinst.run_eflomal()
efinst.run_atools()

Command: ['eflomal-align', '--overwrite', '--input', '/Users/sboisen/git/Clear-Bible/autoalignment/data/eng/BSB/SBLGNT-BSB.piped.txt', '--forward-links', '/Users/sboisen/git/Clear-Bible/alignments-eng/exp/BSB/notebookcheck/forward.pharaoh.txt', '--reverse-links', '/Users/sboisen/git/Clear-Bible/alignments-eng/exp/BSB/notebookcheck/reverse.pharaoh.txt']


FileNotFoundError: [Errno 2] No such file or directory: 'eflomal-align'

### Option: Different Input

You need to supply an `inputname` parameter if you're using a different input file, like lemmas. 

Example:
```
efinst = eflomal.Eflomal(alset, condition="20241209eflomal", inputname="SBLGNT-BSB-lemma.piped.txt")
```

### Option: Using Priors

You can output priors from an initial alignment, and then use those (as-is, or perhaps with editing) in a second alignment pass.

Example:
```
efinst = eflomal.Eflomal(alset, condition="20241209eflomal")
efinst.run_eflomal()
efinst.run_makepriors()
efinst.run_eflomal(readpriors=True)
efinst.run_atools()
```

### Option: Aligning Partial Data

If you've provided a partial corpus as input, the same `eflomal` processes should apply. 

If the partial corpus is small, it could be helpful to train on another larger corpus to generate priors (if one is available): but you should validate this idea with experimentation. 

Alternatively, if you only have partial gold standard data but full source data, you can run full alignment as described above but only generate and score partial _burrito_ data. That generation process is described below; partial scoring is described in the scoring notebook. 

## Regenerating Burrito Data

Once you've created pharaoh-format alignments, they need to be converted back to Scripture Burrito format for consistency in scoring. `make_burrito()` reads the pharaoh file, creates an internal `AlignmentGroup` instance, and then writes it out. 

In [ ]:
pr = reader.PharaohReader(pm)
pr.make_burrito(condition="notebookcheck")

### Regenerating Partial Burrito Data

If you only have partial gold standard data, the best course may be to align the _whole_ corpus (so you have more training data), but then only score the subset of the corpus that corresponds to the available gold standard data. 


### Regenerating Burrito Data from Partial Data

If you've aligned only a partial corpus, ... TBD

## Next Steps

At this point you've created hypothesis data in a compatible Burrito format. The next step is to score the results: see [Scoring Alignment Data](31ScoringAlignmentData.ipynb) for this process. 